# [SK 07 - AI Foundry Agents with Semantic Kernel](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python)
**Note**: `azure-ai-agents==1.1.0b4` and `azure-ai-projects==1.0.0` are automatically installed by semantic kernel 1.35.0.<br/>

An AzureAIAgent is a specialized agent within the Semantic Kernel framework, designed to provide advanced conversational capabilities with seamless tool integration. It automates tool calling, eliminating the need for manual parsing and invocation. The agent also securely manages conversation history using threads, reducing the overhead of maintaining state. Additionally, the AzureAIAgent supports a variety of built-in tools, including file retrieval, code execution, and data interaction via Bing, Azure AI Search, Azure Functions, and OpenAPI.

To use an AzureAIAgent, an Azure AI Foundry Project must be utilized. 

# Constants and Libraries

import os

# Login with tenant ID
os.system("az login --tenant 3ad0b905-34ab-4116-93d9-c1dcc2d35af6 --output none") # --use-device-code

# Set the subscription programmatically
os.system("az account set --subscription eca2eddb-0f0c-4351-a634-52751499eeea")

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-PYTHON"

instructions  = "you are a clever agent"

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"]
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# Create AI FOUNDRY PROJECT CLIENT using Semantic Kernel SDK

In [2]:
from semantic_kernel.agents import AzureAIAgent, AzureAIAgentSettings
os.environ["AZURE_AI_AGENT_ENDPOINT"] = os.environ["AIF_STD_PROJECT_ENDPOINT"]
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  os.environ["MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

client = AzureAIAgent.create_client(credential=DefaultAzureCredential())
AzureAIAgentSettings() # other than "from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings"

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None)

# Native Plugin

In [3]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Create an AI Foundry Agent

## Async version

## Synchronous version

### Retrieve or Create an agent -called `agent_definition`- on the Azure AI agent service

In [4]:
agent_id = "" # "asst_qLugGQ3nZ0wdWgqS6SwZHGVg"

if agent_id != "":
    agent_definition = await client.agents.get_agent(agent_id=agent_id)
else:
    agent_definition = await client.agents.create_agent(
    model=AzureAIAgentSettings().model_deployment_name,
    name=agent_name,
    instructions=instructions,
    )
    
agent_definition

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Failed to invoke the Azure CLI
	AzurePowerShellCredential: Az.Account module >= 2.2.0 is not installed
	AzureDeveloperCliCredential: Please run 'azd auth login' from a command prompt to authenticate before using this credential.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Failed to invoke the Azure CLI
	AzurePowerShellCredential: Az.Account module >= 2.2.0 is not installed
	AzureDeveloperCliCredential: Please run 'azd auth login' from a command prompt to authenticate before using this credential.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.

### Create a Semantic Kernel agent based on `agent_definition`

In [ ]:
agent = AzureAIAgent(
    client=client,
    definition=agent_definition,
    plugins=[LightsPlugin()]
)

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [ ]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Hello", 
    "Please toggle the porch light", 
    "What's the status of all lights?", 
    "Thank you",
]

thread: AzureAIAgentThread = AzureAIAgentThread(client=client)

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        print(f"Message {i} from {AuthorRole.USER}: '{user_input}'")
        response = await agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        await thread.delete()
        print(f"\nthread <{thread.id}> has been deleted.")

In [ ]:
agent.